In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_validation import aligned_returns
from src.research_validation import market_regression
from src.research_data import read_series


# 09 Systematic Market Risk Diagnostics

Use one frozen benchmark and one excess-return alignment for both market-risk diagnostics and Module 12 alpha inference. No live benchmark download or strategy rerun occurs.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Full period market regression

HAC uses the configured lag count. Risk-free conversion uses the prior-session annual-effective rate.


In [ ]:
aligned = aligned_returns(
    pd.read_parquet("equity_curve.parquet"),
    read_series("data/inputs/sp500_prices.parquet"),
    read_series("data/inputs/risk_free_rates.parquet"),
)
aligned.to_parquet("market_alignment.parquet")
market_alpha = market_regression(aligned, cfg.hac_lags)
pd.to_pickle(market_alpha, "market_alpha.pkl")
display(pd.Series(market_alpha))
display(aligned.head())


## 3. Annual and conditional market exposure

Annual and up/down estimates are descriptive subsamples. They do not formally test coefficient equality.


In [ ]:
records = []
for year, group in aligned.groupby(aligned.index.year):
    records.append({"sample": str(year), **market_regression(group, cfg.hac_lags)})
for label, group in [
    ("market_up", aligned.loc[aligned.market_return > 0]),
    ("market_down", aligned.loc[aligned.market_return <= 0]),
]:
    records.append({"sample": label, **market_regression(group, cfg.hac_lags)})
market_subsamples = pd.DataFrame(records)
market_subsamples.to_parquet("market_subsamples.parquet")
display(market_subsamples)


## 4. Rolling beta and correlation

Rolling estimates are descriptive and require 63 observations.


In [ ]:
rolling = pd.DataFrame(index=aligned.index)
rolling["beta_63"] = (
    aligned.strategy_excess.rolling(63).cov(aligned.market_excess)
    / aligned.market_excess.rolling(63).var()
)
rolling["correlation_63"] = aligned.strategy_return.rolling(63).corr(aligned.market_return)
rolling.to_parquet("rolling_market_exposure.parquet")
rolling.plot(
    subplots=True,
    figsize=(11, 5),
    title=["Rolling 63-session beta", "Rolling 63-session correlation"],
)
plt.tight_layout()
plt.show()


## 5. Stress days and relative performance

Use identical aligned dates for the descriptive comparison.


In [ ]:
display(aligned.nsmallest(10, "market_return"))
display(aligned.nsmallest(10, "strategy_return"))
(1 + aligned[["strategy_return", "market_return"]]).cumprod().plot(
    figsize=(11, 4), title="Compounded aligned returns"
)
plt.show()
